# 02 - Feature engineering test (Phase 3 prototyping)
Test window functions, date arithmetic and bucketing on small samples **before** they go into `sql/feature_store.sql`.

Run the pipeline (`python main.py --phase 3`) first if you want to inspect the final `feature_store.olist_features` view.

In [ ]:
import sys
sys.path.insert(0, "..")          # make config.py importable from notebooks/

import psycopg2
from config import DB_CONFIG, load_queries

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

def show(query, limit=20):
    """Run a query and print rows as a plain text table (no DataFrames)."""
    cur.execute(query)                       # no params -> '%' in SQL is safe
    header = [c[0] for c in cur.description]
    rows = cur.fetchmany(limit)
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) if rows else len(str(h))
              for i, h in enumerate(header)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(header, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(str(v).ljust(w) for v, w in zip(r, widths)))

## 1. Customer history with window functions (no leakage: frame ends at `1 PRECEDING`)

In [ ]:
show("""
SELECT cu.customer_unique_id,
       o.order_purchase_timestamp,
       COUNT(*) OVER (PARTITION BY cu.customer_unique_id
                      ORDER BY o.order_purchase_timestamp, o.order_id
                      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prior_orders,
       LAG(o.order_purchase_timestamp) OVER (PARTITION BY cu.customer_unique_id
                      ORDER BY o.order_purchase_timestamp, o.order_id) AS prev_order_ts
FROM orders o
JOIN customers cu ON cu.customer_id = o.customer_id
WHERE cu.customer_unique_id IN (
    SELECT customer_unique_id FROM customers
    GROUP BY customer_unique_id HAVING COUNT(*) > 2 LIMIT 3)
ORDER BY cu.customer_unique_id, o.order_purchase_timestamp
""")

## 2. Date arithmetic: delivery deltas

In [ ]:
show("""
SELECT order_id,
       ROUND(EXTRACT(EPOCH FROM (order_estimated_delivery_date - order_purchase_timestamp)) / 86400.0, 1) AS estimated_days,
       ROUND(EXTRACT(EPOCH FROM (order_delivered_customer_date - order_purchase_timestamp)) / 86400.0, 1) AS actual_days,
       CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late_delivery
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
LIMIT 10
""")

## 3. Haversine distance (km) between two points

In [ ]:
show("""
SELECT ROUND((6371.0 * 2 * ASIN(SQRT(
         POWER(SIN(RADIANS(-22.9068 - (-23.5505)) / 2), 2)
       + COS(RADIANS(-23.5505)) * COS(RADIANS(-22.9068))
       * POWER(SIN(RADIANS(-43.1729 - (-46.6333)) / 2), 2))))::NUMERIC, 1) AS sao_paulo_to_rio_km
""")

## 4. High-cardinality bucketing with `CASE WHEN` + regex

In [ ]:
show("""
SELECT category_group, COUNT(*) AS products
FROM (
    SELECT CASE
             WHEN product_category_name ~ '^(bed_bath|furniture|home|housewares)' THEN 'home_and_furniture'
             WHEN product_category_name ~ '^(computers|telephony|electronics)' THEN 'electronics_and_tech'
             WHEN product_category_name IS NULL THEN 'unknown'
             ELSE 'other'
           END AS category_group
    FROM products
) t
GROUP BY category_group
ORDER BY products DESC
""")

## 5. Percentile capping (winsorisation) preview

In [ ]:
show("""
SELECT PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY price) AS p01,
       PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY price) AS p99,
       MAX(price) AS raw_max
FROM order_items
""")

## 6. Inspect the final Feature Store (after running phase 3)

In [ ]:
show("SELECT * FROM feature_store.olist_features LIMIT 5", limit=5)

In [ ]:
cur.close()
conn.close()